In [ ]:
# pyeda requires system-level dependencies (espresso logic minimizer).
# On macOS: brew install espresso
# On Ubuntu/Debian: sudo apt-get install python3-dev
# Then install the Python packages:
!pip install pyeda merlin-xai

In [ ]:
import os
import logging
from zipfile import ZipFile
import pandas as pd
from merlin import MERLIN
import requests
import zipfile
import io


In [ ]:
import requests
import pandas as pd
from zipfile import ZipFile
import io

# URL of the zip file
url = "https://raw.githubusercontent.com/Crisp-Unimib/MERLIN/main/tests/test_data/20newsgroups.zip"

# Download the zip file
response = requests.get(url)

# Check if the file was downloaded successfully
if response.status_code == 200:
    print("File downloaded successfully!")
    
    # Open the zip file from memory
    with ZipFile(io.BytesIO(response.content), 'r') as archive:
        # Read CSV files directly from the archive
        df_left = pd.read_csv(archive.open('df_left.csv'), delimiter=',')
        df_right = pd.read_csv(archive.open('df_right.csv'), delimiter=',')

    # Filter out rows where 'corpus' is null
    df_left = df_left[~df_left['corpus'].isnull()]
    df_right = df_right[~df_right['corpus'].isnull()]

    # Load data
    X_left, Y_left, predicted_labels_left = (
        df_left['corpus'], df_left['category'], df_left['predicted_labels']
    )
    X_right, Y_right, predicted_labels_right = (
        df_right['corpus'], df_right['category'], df_right['predicted_labels']
    )
    
    print("Data loaded and processed successfully!")
else:
    print(f"Failed to download file. Status code: {response.status_code}")

In [ ]:
df_left

In [ ]:
exp = MERLIN(X_left, predicted_labels_left,
             X_right, predicted_labels_right,
             data_type='text', surrogate_type='sklearn', log_level=logging.INFO,
             hyperparameters_selection=True, save_path=f'results/',
             save_surrogates=True, save_bdds=True)

percent_dataset = 1
print(
    f'Running Trace with percent of dataset to use: {percent_dataset}', flush=True)
exp.run_trace(percent_dataset)

In [ ]:
exp.run_explain()


In [ ]:
exp.explain.BDD2Text()